# Llama 3.1 8B QLoRA Fine-Tuning for ERP Finance
This notebook uses the supplied ERP Finance train, validation, and test files as separate splits. Validation monitors training; the test set is evaluated only after training is complete. Checkpoints and final models are saved to Google Drive.

Company-specific policies and role restrictions remain in the RAG/policy retrieval layer.


In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install rouge-score

### Google Drive and Persistent Training Configuration
Place all three supplied JSONL files in `/content/drive/MyDrive/erp_research/data/`, or upload them to `/content/sample_data/`. The resolver also accepts a repository `Data/` directory.

| Purpose | File | Current rows |
| --- | --- | ---: |
| Training: update LoRA weights | erp_finance_dataset_4000_v2_train_clean.jsonl | 3165 |
| Validation / evaluation: monitor loss and compare development results | erp_finance_dataset_4000_v2_validation_clean.jsonl | 433 |
| Test: final held-out report | erp_finance_dataset_4000_v2_test_clean.jsonl | 402 |

Use the existing validation file; do not split training again or use test examples for checkpoint selection. Do not load the combined 4000-row file as training data. Change `RUN_NAME` for a new experiment; reuse it only to resume the same run.


In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/erp_research"
RUN_NAME = "erp_finance_v2_clean"
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints", RUN_NAME)
FINAL_MODEL_DIR = os.path.join(PROJECT_DIR, "models", RUN_NAME, "lora_final")
FINAL_GGUF_DIR = os.path.join(PROJECT_DIR, "models", RUN_NAME, "gguf_final")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results", RUN_NAME)
DATA_DIR = os.path.join(PROJECT_DIR, "data")

for folder in [CHECKPOINT_DIR, FINAL_MODEL_DIR, FINAL_GGUF_DIR, RESULTS_DIR, DATA_DIR]:
    os.makedirs(folder, exist_ok=True)

def resolve_dataset_file(filename):
    candidates = [
        Path(DATA_DIR) / filename,
        Path("/content/sample_data") / filename,
        Path.cwd() / "Data" / filename,
        Path.cwd().parent / "Data" / filename,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate)
    raise FileNotFoundError(
        f"Dataset not found: {filename}. Place it in {DATA_DIR} "
        "or /content/sample_data/. Checked: "
        + ", ".join(str(path) for path in candidates)
    )

TRAIN_FILE = resolve_dataset_file("erp_finance_dataset_4000_v2_train_clean.jsonl")
VALIDATION_FILE = resolve_dataset_file("erp_finance_dataset_4000_v2_validation_clean.jsonl")
TEST_FILE = resolve_dataset_file("erp_finance_dataset_4000_v2_test_clean.jsonl")

print("Checkpoint directory:", CHECKPOINT_DIR)
print("Training dataset:", TRAIN_FILE)
print("Validation / evaluation dataset:", VALIDATION_FILE)
print("Final test dataset:", TEST_FILE)


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024  # Lower memory use; increase only if your examples are genuinely longer
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

### 📌 Base Model Parameter Stats (Before LoRA)

In [ ]:
# Record GPU memory and parameter count BEFORE LoRA adapters are added
import torch

total_params_base = sum(p.numel() for p in model.parameters())
trainable_params_base = sum(p.numel() for p in model.parameters() if p.requires_grad)

gpu_mem_before_lora = torch.cuda.memory_allocated() / 1e9  # GB

print(f"Total parameters     : {total_params_base:,}")
print(f"Trainable parameters : {trainable_params_base:,}")
print(f"GPU memory (before LoRA): {gpu_mem_before_lora:.2f} GB")


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### ⚡ LoRA Parameter Efficiency

In [ ]:
# Show LoRA impact: how many parameters are actually being trained
total_params_lora = sum(p.numel() for p in model.parameters())
trainable_params_lora = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params_lora = total_params_lora - trainable_params_lora

gpu_mem_after_lora = torch.cuda.memory_allocated() / 1e9  # GB

trainable_pct = (trainable_params_lora / total_params_lora) * 100

print("=" * 55)
print("          LoRA PARAMETER EFFICIENCY REPORT")
print("=" * 55)
print(f"{'Total parameters':<30}: {total_params_lora:>15,}")
print(f"{'Frozen parameters':<30}: {frozen_params_lora:>15,}")
print(f"{'Trainable (LoRA) parameters':<30}: {trainable_params_lora:>15,}")
print(f"{'Trainable %':<30}: {trainable_pct:>14.4f}%")
print("-" * 55)
print(f"{'GPU Memory before LoRA':<30}: {gpu_mem_before_lora:>13.2f} GB")
print(f"{'GPU Memory after LoRA':<30}: {gpu_mem_after_lora:>13.2f} GB")
print(f"{'Memory overhead of LoRA':<30}: {gpu_mem_after_lora - gpu_mem_before_lora:>13.2f} GB")
print("=" * 55)
print(f"\n✅ LoRA trains only {trainable_pct:.4f}% of parameters — full fine-tuning would train 100%.")


### Load and Validate the Three Dataset Splits
Preserve the supplied `system`, `user`, and `assistant` roles and instructions. The validation file becomes the trainer's `eval_dataset`. The test file is excluded from training and validation.

The checks below reject empty or malformed splits and duplicate questions across splits. They check normalized text, not semantic similarity; paraphrased questions can still overlap.


In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
from itertools import combinations

tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

dataset = load_dataset("json", data_files={
    "train": TRAIN_FILE,
    "validation": VALIDATION_FILE,
    "test": TEST_FILE,
})

def normalized_question(record):
    return " ".join(record["user"].split()).casefold()

def validate_splits(splits):
    question_sets = {}
    for name, records in splits.items():
        if not len(records):
            raise ValueError(f"{name} split is empty.")
        for index, record in enumerate(records):
            for field in ("system", "user", "assistant"):
                if not isinstance(record.get(field), str) or not record[field].strip():
                    raise ValueError(f"{name} row {index + 1}: missing/blank {field}.")
        questions = [normalized_question(record) for record in records]
        if len(set(questions)) != len(questions):
            raise ValueError(f"{name} contains duplicate normalized questions.")
        question_sets[name] = set(questions)
    for left, right in combinations(question_sets, 2):
        shared = question_sets[left] & question_sets[right]
        if shared:
            raise ValueError(f"{left} and {right} share {len(shared)} questions.")

validate_splits(dataset)
train_dataset = dataset["train"]
eval_dataset = dataset["validation"]
test_dataset = dataset["test"]

DEFAULT_ERP_SYSTEM_PROMPT = (
    "You are an ERP Finance assistant. Answer only ERP Finance questions. "
    "Do not invent company policies or approval chains. Request current company "
    "context when needed. For unrelated questions, respond with OUT_OF_SCOPE."
)

def format_prompts(examples):
    conversations = []

    for system_msg, user_msg, assistant_msg in zip(
        examples["system"], examples["user"], examples["assistant"]
    ):
        # Preserve dataset roles correctly. Use the broad ERP prompt only when
        # the dataset system field is missing/blank.
        system_content = (
            system_msg.strip()
            if isinstance(system_msg, str) and system_msg.strip()
            else DEFAULT_ERP_SYSTEM_PROMPT
        )
        user_content = user_msg.strip() if isinstance(user_msg, str) else str(user_msg)
        assistant_content = assistant_msg.strip() if isinstance(assistant_msg, str) else str(assistant_msg)

        convo = [
            {"role": "system", "content": system_content},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ]
        conversations.append(convo)

    formatted_texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        )
        for convo in conversations
    ]

    return {"text": formatted_texts}

train_dataset = train_dataset.map(format_prompts, batched=True)
eval_dataset = eval_dataset.map(format_prompts, batched=True)
test_dataset = test_dataset.map(format_prompts, batched=True)

print(f"Train size: {len(train_dataset)}")
print(f"Validation / eval size: {len(eval_dataset)}")
print(f"Test size: {len(test_dataset)}")
print("Example formatted record:\n")
print(train_dataset[0]["text"][:1500])


### Base Model Validation Loss
Compute loss and perplexity on all validation examples, with LoRA disabled. The fine-tuned comparison uses exactly the same rows and loss calculation. These are full-conversation token losses, including the prompt, not answer-only correctness scores.


In [ ]:
import math
import torch

def compute_eval_loss(mdl, tok, dataset):
    """Token-weighted next-token loss on every row, using the training length limit."""
    total_loss, total_tokens = 0.0, 0
    for sample in dataset:
        inputs = tok(
            sample["text"], return_tensors="pt", add_special_tokens=False,
            truncation=True, max_length=max_seq_length,
        ).to("cuda")
        token_count = inputs["input_ids"].shape[-1] - 1
        if token_count <= 0:
            raise ValueError("An evaluation record has no prediction targets.")
        with torch.no_grad():
            out = mdl(**inputs, labels=inputs["input_ids"])
        if out.loss is None or not torch.isfinite(out.loss).item():
            raise ValueError("Non-finite evaluation loss.")
        total_loss += out.loss.item() * token_count
        total_tokens += token_count
    if not total_tokens:
        raise ValueError("Cannot evaluate an empty dataset.")
    return total_loss / total_tokens

FastLanguageModel.for_inference(model)
model.eval()
print(f"Evaluating BASE model on {len(eval_dataset)} validation rows...")
with model.disable_adapter():
    base_eval_loss = compute_eval_loss(model, tokenizer, eval_dataset)
base_perplexity = math.exp(base_eval_loss)
print(f"Validation loss: {base_eval_loss:.4f}; perplexity: {base_perplexity:.2f}")
FastLanguageModel.for_training(model)


In [ ]:
# Development evaluation uses validation only. The test set is used in the final section.
import json
import torch
from rouge_score import rouge_scorer as rs_module

# Exploratory lexical-overlap threshold, not a calibrated factual-accuracy threshold.
# Keep it fixed before inspecting test results.
ROUGE_THRESHOLD = 0.35

def reference_is_out_of_scope(text):
    return text.strip().upper().startswith("OUT_OF_SCOPE")

def check_refusal(text):
    # Explicit scope language avoids treating "I cannot find the company policy" as out of scope.
    normalized = " ".join(text.casefold().replace("_", " ").replace("-", " ").split())
    phrases = (
        "out of scope", "outside the scope", "outside my scope",
        "not related to erp", "unrelated to erp",
        "only assist with questions related to erp finance",
        "only answer questions related to erp finance",
    )
    return any(phrase in normalized for phrase in phrases)

eval_records = list(dataset["validation"])

def describe_records(records, label):
    out_count = sum(reference_is_out_of_scope(row["assistant"]) for row in records)
    print(f"{label}: {len(records)} rows; {len(records) - out_count} in scope; "
          f"{out_count} out of scope.")

describe_records(eval_records, "Validation")
print("ROUGE-L threshold:", ROUGE_THRESHOLD)
print("Scores are automated proxies; review factual answers and refusals manually.")

def run_erp_eval(mdl, tok, records, max_new_tokens=256, label="", threshold=ROUGE_THRESHOLD):
    """ROUGE/refusal proxy scores, with per-question output for manual review."""
    if not len(records):
        raise ValueError("Cannot evaluate an empty split.")
    mdl.eval()
    scorer = rs_module.RougeScorer(["rougeL"], use_stemmer=True)
    results = []

    for i, sample in enumerate(records):
        reference = sample["assistant"].strip()
        is_out = reference_is_out_of_scope(reference)
        messages = [
            {"role": "system", "content": sample["system"]},
            {"role": "user", "content": sample["user"]},
        ]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, add_special_tokens=False, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out_ids = mdl.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False, use_cache=True,
            )
        generated = tok.decode(
            out_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True
        ).strip()
        rouge_l = scorer.score(reference, generated)["rougeL"].fmeasure
        refused = check_refusal(generated)
        correct = refused if is_out else (not refused and rouge_l >= threshold)
        results.append({
            "user": sample["user"], "reference": reference,
            "generated": generated, "rouge_l": rouge_l,
            "correct": correct, "out_of_scope": is_out,
        })
        if (i + 1) % 25 == 0 or i + 1 == len(records):
            print(f"[{label}] {i + 1}/{len(records)} evaluated")

    in_scope = [row for row in results if not row["out_of_scope"]]
    out_scope = [row for row in results if row["out_of_scope"]]
    def pass_rate(rows):
        return sum(row["correct"] for row in rows) / len(rows) * 100 if rows else None
    return {
        "accuracy": pass_rate(results),
        "in_scope_acc": pass_rate(in_scope),
        "out_scope_acc": pass_rate(out_scope),
        "avg_rouge_l": sum(row["rouge_l"] for row in results) / len(results),
        "results": results, "n": len(results), "threshold": threshold,
    }

print("Validation evaluation helper ready.")


In [ ]:
print(f"Evaluating BASE model on {len(eval_records)} validation questions...")
FastLanguageModel.for_inference(model)
# PEFT's context manager restores the adapter after the base-model pass.
# https://huggingface.co/docs/peft/package_reference/peft_model#peft.PeftModel.disable_adapter
with model.disable_adapter():
    base_results = run_erp_eval(model, tokenizer, eval_records, label="BASE VALIDATION")

base_accuracy = base_results["accuracy"]
base_rouge_l = base_results["avg_rouge_l"]
print(f"Base validation proxy score: {base_accuracy:.2f}%")
print(f"Base validation ROUGE-L: {base_rouge_l:.4f}")
FastLanguageModel.for_training(model)


### Train with Validation
Only `train_dataset` updates weights. `eval_dataset` is the supplied validation file. Select the best checkpoint by validation loss; never by test scores.


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from transformers.trainer_utils import get_last_checkpoint
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    # Keep validation examples independent.
    # Packing can be tested later as a separate speed optimization.
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=25,

        # Save frequently to persistent Google Drive storage.
        save_strategy="steps",
        save_steps=25,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=CHECKPOINT_DIR,
        report_to="none",
    ),
)

# Automatically continue from the newest checkpoint after a Colab disconnect.
latest_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)

if latest_checkpoint:
    print("Resuming training from:", latest_checkpoint)
    trainer_stats = trainer.train(resume_from_checkpoint=latest_checkpoint)
else:
    print("No checkpoint found. Starting a new training run.")
    trainer_stats = trainer.train()

print("Training finished.")

### Fine-Tuned Validation Loss
Re-evaluate the selected checkpoint on the complete validation set using the same function as the baseline.


In [ ]:
FastLanguageModel.for_inference(model)
model.eval()
ft_eval_loss = compute_eval_loss(model, tokenizer, eval_dataset)
ft_perplexity = math.exp(ft_eval_loss)

print(f"{'Validation metric':<24} {'Base':>12} {'Fine-tuned':>12}")
print(f"{'Full-conversation loss':<24} {base_eval_loss:>12.4f} {ft_eval_loss:>12.4f}")
print(f"{'Perplexity':<24} {base_perplexity:>12.2f} {ft_perplexity:>12.2f}")
print("These metrics measure next-token prediction, not factual ERP correctness.")


### Fine-Tuned Validation Generation Scores
Use validation results for development. ROUGE-L measures text overlap and the refusal check detects scope language; neither guarantees factual correctness or a complete refusal.


In [ ]:
FastLanguageModel.for_inference(model)
ft_results = run_erp_eval(model, tokenizer, eval_records, label="FT VALIDATION")
ft_accuracy = ft_results["accuracy"]
ft_rouge_l = ft_results["avg_rouge_l"]

def print_comparison(base, tuned, label):
    print(f"\n{label}: {base['n']} questions")
    print(f"{'Metric':<30} {'Base':>12} {'Fine-tuned':>12}")
    for key, title in [
        ("accuracy", "Overall proxy score (%)"),
        ("in_scope_acc", "In-scope ROUGE pass (%)"),
        ("out_scope_acc", "Scope refusal detected (%)"),
        ("avg_rouge_l", "Average ROUGE-L"),
    ]:
        left = f"{base[key]:.4f}" if base[key] is not None else "N/A"
        right = f"{tuned[key]:.4f}" if tuned[key] is not None else "N/A"
        print(f"{title:<30} {left:>12} {right:>12}")

print_comparison(base_results, ft_results, "VALIDATION")


### Final Held-Out Test
Run this section after training and checkpoint selection are complete. Compare the base model (LoRA disabled) and the selected fine-tuned model on the same 402 test examples. Do not use these scores to tune this experiment.

The base pass uses the [PEFT adapter-disable context manager](https://huggingface.co/docs/peft/package_reference/peft_model#peft.PeftModel.disable_adapter). Reports include generated answers for manual review.


In [ ]:
test_records = list(dataset["test"])
describe_records(test_records, "Final test")
FastLanguageModel.for_inference(model)
model.eval()

with model.disable_adapter():
    base_test_loss = compute_eval_loss(model, tokenizer, test_dataset)
    base_test_results = run_erp_eval(model, tokenizer, test_records, label="BASE TEST")

ft_test_loss = compute_eval_loss(model, tokenizer, test_dataset)
ft_test_results = run_erp_eval(model, tokenizer, test_records, label="FT TEST")
print_comparison(base_test_results, ft_test_results, "FINAL TEST")
print(f"Test loss: base={base_test_loss:.4f}, fine-tuned={ft_test_loss:.4f}")
print(f"Test perplexity: base={math.exp(base_test_loss):.2f}, "
      f"fine-tuned={math.exp(ft_test_loss):.2f}")

report = {
    "files": {"train": TRAIN_FILE, "validation": VALIDATION_FILE, "test": TEST_FILE},
    "sizes": {name: len(rows) for name, rows in dataset.items()},
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "metric_note": "ROUGE/refusal scores are proxies, not verified factual accuracy.",
    "validation": {
        "base_loss": base_eval_loss, "fine_tuned_loss": ft_eval_loss,
        "base": base_results, "fine_tuned": ft_results,
    },
    "test": {
        "base_loss": base_test_loss, "fine_tuned_loss": ft_test_loss,
        "base": base_test_results, "fine_tuned": ft_test_results,
    },
}
report_path = os.path.join(RESULTS_DIR, "evaluation_report.json")
with open(report_path, "w", encoding="utf-8") as fh:
    json.dump(report, fh, ensure_ascii=False, indent=2, allow_nan=False)
print("Saved evaluation report:", report_path)


In [ ]:
# 2. Qualitative Evaluation (Inference)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "system", "content": "You are a domain-aware AI assistant for a traditional ERP system. Answer ERP process questions clearly using the provided context."},
    {"role": "user", "content": "Classify the following into an ERP transaction.\nContext: Bought office supplies for $500."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 64, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])

### Save the Model to Google Drive
The LoRA adapter, tokenizer, and optional GGUF export are saved in Drive so they remain available after the Colab runtime ends.

In [ ]:
# Save final LoRA adapter and tokenizer to persistent Google Drive
model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print("Saved LoRA adapter and tokenizer to:", FINAL_MODEL_DIR)

# Export a 4-bit GGUF for Ollama to persistent Google Drive.
# This can take additional time and Drive space. Comment this block out if you only need the LoRA adapter.
model.save_pretrained_gguf(
    FINAL_GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)
print("Saved GGUF export to:", FINAL_GGUF_DIR)

In [ ]:
from google.colab import files
import os

# Optional: download one GGUF file to your computer.
# The model is already safe in Google Drive, so downloading is not required.
if os.path.exists(FINAL_GGUF_DIR):
    gguf_files = [f for f in os.listdir(FINAL_GGUF_DIR) if f.endswith(".gguf")]
    if gguf_files:
        gguf_file = os.path.join(FINAL_GGUF_DIR, gguf_files[0])
        print("GGUF stored safely in Drive:", gguf_file)
        print("If you want a local copy, uncomment the next line:")
        print(f'# files.download("{gguf_file}")')
    else:
        print("No GGUF file found in:", FINAL_GGUF_DIR)
else:
    print("GGUF directory does not exist:", FINAL_GGUF_DIR)